# ML Ops lab

To download this lab material click on the [link](https://github.com/SETU-AIML-2026/DH-Infra-AI/tree/main/topic-04-sprint-3-AI/unit-2-ml-ops/notebook-1-mlops/lab8.ipynb)

Switch to T4 GPU Runtime:

    * In the top-right menu of this Colab page, click the arrow next to the Connect button (or click Runtime in the top menu bar).
    * Select Change runtime type.
    * Under Hardware accelerator, choose T4 GPU.
    * Click Save.

Accessing the Data:

    * Make sure you have a reliable internet connection so that the public URL (https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv) is accessible during training. See previous lab materials on how to split the data for training.
    * If you still have it, you can load it from Google Cloud Storage (GCS), ensure your GCS bucket permissions allow access, and authenticate using Google Cloud SDK tools in the notebook.



### Load Data, Train and Compare Models

# Imports
this part should be in a requirements.txt with exact version numbers!

In [ ]:
!pip install mlflow

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import warnings
import time
from google.cloud import storage
from google.colab import auth
import mlflow
import mlflow.pytorch
import itertools
warnings.filterwarnings('ignore')

### Model Hyperparameter Configuration

In [ ]:
# Define hyperparameters for SVM and Deep Neural Network
SVM_RANDOM_STATE = 42
SVM_MAX_ITER = 1000

# PyTorch Neural Network Hyperparameters
NN_INPUT_SIZE = 8  # Number of preprocessed features (Pclass, Age, SibSp, Parch, Fare, Sex_male, Embarked_Q, Embarked_S)
NN_HIDDEN_SIZE = 1000
NN_EPOCHS = 100
NN_BATCH_SIZE = 64
NN_LEARNING_RATE = 0.001
NN_RANDOM_STATE = 42

# Device selection
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

In [ ]:
# Set random seeds for reproducibility
torch.manual_seed(NN_RANDOM_STATE)
np.random.seed(NN_RANDOM_STATE)

# Connecting to GCS
The project and bucket name should be in an enviromental variable

UPDATE THIS PART TO MATCH YOUR BUCKET OR RE-CREATE THE DATA FROM THE RAW ONE (see [LAB3](https://tutors.dev/notebook/dh-infra-ai/topic-02-data-storage/unit-3-databases/notebook-1-gcs))


In [ ]:
# --- GCP Configuration ---
PROJECT_ID = "Dh-inf-for-AI"
BUCKET_NAME = "titanic-dh-data"

auth.authenticate_user()

In [ ]:
# Load the Titanic dataset splits from GCS sharded folder
train_url = f"gs://{BUCKET_NAME}/sharded/train.csv"
dev_url = f"gs://{BUCKET_NAME}/sharded/dev.csv"
test_url = f"gs://{BUCKET_NAME}/sharded/test.csv"

# Reading the files using pandas
df_train = pd.read_csv(train_url)
df_dev = pd.read_csv(dev_url)
df_test = pd.read_csv(test_url)

print(f"Loaded train shape: {df_train.shape}, dev shape: {df_dev.shape}, test shape: {df_test.shape}")

In [ ]:
# Preparing features
features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex', 'Embarked']

# Helper function to process each dataset split consistently
def preprocess_split(df_split):
    df_processed = df_split[features].copy()

    # Fill missing values using training set statistics or defaults
    df_processed['Age'].fillna(df_train['Age'].median(), inplace=True)
    df_processed['Fare'].fillna(df_train['Fare'].median(), inplace=True)
    df_processed['Embarked'].fillna(df_train['Embarked'].mode()[0], inplace=True)

    # Convert categorical features to numeric
    df_processed = pd.get_dummies(df_processed, columns=['Sex', 'Embarked'], drop_first=True)

    # Ensure both splits have the exact same columns after dummy encoding
    expected_columns = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_Q', 'Embarked_S']
    for col in expected_columns:
        if col not in df_processed.columns:
            df_processed[col] = 0
    return df_processed[expected_columns]

# Prepare features for both Train and Dev splits
X_train = preprocess_split(df_train)
y_train = df_train['Survived']

X_test = preprocess_split(df_dev)
y_test = df_dev['Survived']

print(f"Final shapes - X_train: {X_train.shape}, X_test (dev): {X_test.shape}")

In [ ]:
#Feature Normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# AI training

In [ ]:
# Define PyTorch MLP Model Architecture
class TitanicMLP(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(TitanicMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, hidden_size),
            nn.Sigmoid(),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# Explicitly set tracking URI and enable the file store backend
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.makedirs("/content/mlruns_clean", exist_ok=True)
#if you set a remote tracking server here then you can inspect the results via web UI too (see future tasks part at the end)
mlflow.set_tracking_uri("file:///content/mlruns_clean")

# Ensure the active experiment is created/set before running any start_run calls
mlflow.set_experiment("Titanic_Grid_Search")

def train_and_eval_pytorch_mlflow(X_tr, y_tr, X_te, y_te, device, hidden_size, lr, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        start_time = time.time()

        # Convert data to PyTorch Tensors and cast to float32
        X_tr_t = torch.FloatTensor(np.array(X_tr).astype(np.float32)).to(device)
        y_tr_t = torch.FloatTensor(np.array(y_tr).astype(np.float32)).unsqueeze(1).to(device)
        X_te_t = torch.FloatTensor(np.array(X_te).astype(np.float32)).to(device)

        dataset = TensorDataset(X_tr_t, y_tr_t)
        loader = DataLoader(dataset, batch_size=NN_BATCH_SIZE, shuffle=True)

        # Instantiate model, loss, and optimizer
        model = TitanicMLP(X_tr.shape[1], hidden_size).to(device)
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        # Log hyperparameters to MLflow
        mlflow.log_param("input_size", X_tr.shape[1])
        mlflow.log_param("hidden_size", hidden_size)
        mlflow.log_param("epochs", NN_EPOCHS)
        mlflow.log_param("batch_size", NN_BATCH_SIZE)
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("device", str(device))

        loss_history = []

        # Training loop
        model.train()
        for epoch in range(NN_EPOCHS):
            epoch_loss = 0.0
            for batch_x, batch_y in loader:
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * batch_x.size(0)

            avg_epoch_loss = epoch_loss / len(X_tr)
            loss_history.append(avg_epoch_loss)
            mlflow.log_metric("loss", avg_epoch_loss, step=epoch)

        # Evaluation
        model.eval()
        with torch.no_grad():
            predictions = model(X_te_t).cpu().numpy()
            predictions = (predictions > 0.5).astype(int)
            acc = accuracy_score(y_te, predictions)

        duration = time.time() - start_time

        # Log final metrics and model
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("duration_seconds", duration)
        mlflow.pytorch.log_model(model, "model", serialization_format="pickle")

        print(f"[{run_name}] Accuracy: {acc:.4f}, Duration: {duration:.2f}s")

    return acc, loss_history, model, duration

# Define Grid Search Parameters
hidden_sizes = [100, 500, 1000]
learning_rates = [0.1, 0.01, 0.001]

best_acc = 0.0
best_params = {}
best_model = None

print("=== Starting Grid Search with MLflow Logging ===")
for hs, lr in itertools.product(hidden_sizes, learning_rates):
    # Include formatted timestamp in the run name
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_title = f"Titanic_MLP_HS_{hs}_LR_{lr}_{timestamp}"
    torch.manual_seed(NN_RANDOM_STATE)
    np.random.seed(NN_RANDOM_STATE)
    acc, loss_hist, model, duration = train_and_eval_pytorch_mlflow(
        X_train_scaled, y_train, X_test_scaled, y_test, DEVICE, hs, lr, run_name=run_title
    )
    if acc > best_acc:
        best_acc = acc
        best_params = {"hidden_size": hs, "learning_rate": lr}
        best_model = model

print("\n=== Best Hyperparameter Set Found ===")
print(f"Best Parameters: {best_params}")
print(f"Best Accuracy:   {best_acc:.4f}")

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

# 1. Search and retrieve the most recent MLflow run
runs_df = mlflow.search_runs(order_by=["attribute.start_time DESC"], max_results=1)

if not runs_df.empty:
    latest_run = runs_df.iloc[0]
    run_id = latest_run["run_id"]
    run_name = latest_run["tags.mlflow.runName"]

    print(f"=== Latest MLflow Run: {run_name} (ID: {run_id}) ===\n")

    # 2. Access logged Hyperparameters
    print("--- Logged Parameters ---")
    params = {k.replace("params.", ""): v for k, v in latest_run.items() if k.startswith("params.")}
    for param_name, param_val in params.items():
        print(f"{param_name}: {param_val}")

    # 3. Access logged final Metrics
    print("\n--- Logged Summary Metrics ---")
    metrics = {k.replace("metrics.", ""): v for k, v in latest_run.items() if k.startswith("metrics.")}
    for metric_name, metric_val in metrics.items():
        if metric_name != "loss": # loss is logged per-step, we'll fetch its history below
            print(f"{metric_name}: {metric_val}")

    # 4. Access full metric history (e.g. Training Loss per epoch)
    client = MlflowClient()
    loss_history = client.get_metric_history(run_id, "loss")

    print("\n--- Training Loss History (First 5 and Last 5 Epochs) ---")
    print(f"Total epochs logged: {len(loss_history)}")

    # Display head of loss history
    for i, metric in enumerate(loss_history[:5]):
        print(f"  Epoch {metric.step}: Loss = {metric.value:.4f}")
    print("  ...")
    # Display tail of loss history
    for i, metric in enumerate(loss_history[-5:]):
        print(f"  Epoch {metric.step}: Loss = {metric.value:.4f}")
else:
    print("No MLflow runs found. Make sure you run the training cell first to log data!")

# Practice tasks

- Task 1: Plot MLflow Learning/Loss Curves
- Task 2: Compare Model Training Times (which hyperparameters lead to the fastest training?)
- Task 3: Configure and Launch MLflow GUI Server (either [locally](https://mlflow.org/docs/latest/genai/getting-started/connect-environment/) or use a service like [DataBricks](https://docs.databricks.com/aws/en/mlflow/), [DagsHub](https://dagshub.com/))

## Let's do the same thing with another tool (Tensorboard)

In [ ]:
from torch.utils.tensorboard import SummaryWriter
import itertools

# 1. Load the TensorBoard notebook extension
%load_ext tensorboard

def train_and_eval_pytorch_tb_grid(X_tr, y_tr, X_te, y_te, device, hidden_sizes, learning_rates, epochs=NN_EPOCHS):
    # Convert dataset to PyTorch Tensors
    X_tr_t = torch.FloatTensor(np.array(X_tr).astype(np.float32)).to(device)
    y_tr_t = torch.FloatTensor(np.array(y_tr).astype(np.float32)).unsqueeze(1).to(device)
    X_te_t = torch.FloatTensor(np.array(X_te).astype(np.float32)).to(device)

    dataset = TensorDataset(X_tr_t, y_tr_t)

    print("=== Starting Grid Search with TensorBoard Logging ===")
    for hs, lr in itertools.product(hidden_sizes, learning_rates):
        run_name = f"Titanic_MLP_HS_{hs}_LR_{lr}"
        print(f"Running: {run_name}")

        # Initialize the SummaryWriter for this run's subdirectory
        writer = SummaryWriter(f'runs/{run_name}')
        loader = DataLoader(dataset, batch_size=NN_BATCH_SIZE, shuffle=True)

        #set the seed
        torch.manual_seed(NN_RANDOM_STATE)
        np.random.seed(NN_RANDOM_STATE)

        # Instantiate model, loss, and optimizer
        model = TitanicMLP(X_tr.shape[1], hs).to(device)
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(epochs):
            model.train()
            epoch_loss = 0.0
            for batch_x, batch_y in loader:
                optimizer.zero_grad()
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item() * batch_x.size(0)

            avg_epoch_loss = epoch_loss / len(X_tr)

            # Evaluate current epoch accuracy on validation set
            model.eval()
            with torch.no_grad():
                predictions = model(X_te_t).cpu().numpy()
                predictions = (predictions > 0.5).astype(int)
                val_acc = accuracy_score(y_te, predictions)

            # Log metrics to TensorBoard
            writer.add_scalar('Loss/train', avg_epoch_loss, epoch)
            writer.add_scalar('Accuracy/val', val_acc, epoch)

        # Close the writer for the current configuration run
        writer.close()

    print("Grid search finished! Launch TensorBoard below to inspect and compare the runs.")

# Run the PyTorch grid search with TensorBoard logging
hidden_sizes = [100, 500, 1000]
learning_rates = [0.1, 0.01, 0.001]

train_and_eval_pytorch_tb_grid(
    X_train_scaled, y_train, X_test_scaled, y_test, DEVICE, hidden_sizes, learning_rates, epochs=50
)

In [ ]:
# Launch TensorBoard inline inside Google Colab
# Sometimes it takes a bit of time to load the UI so be patient
%tensorboard --logdir runs

# Now let's try to draw some conclusions based on what you see:

Are there signs of under/overfitting?

What could be the reason? (which hyperparameter could be the issue)

Which hyperparameter turned out to be the most important one? (i.e. changing it caused big changes in accuracy/loss)

Are the metrics (loss/accuracy) strongly correlated?

### Future Tasks

* **External MLOps Tracking Integration**:
  * Configure MLflow to log training runs, parameters, and performance metrics to an external server like **Databricks** or **Dagshub**.
  * Alternatively integrate **Weights & Biases (W&B)** to track experiments, visualize interactive training curves, and monitor resource consumption.

* **Model Registry & Deployment**:
  * Register the best-performing models directly to the MLflow Model Registry or Databricks Unity Catalog.
  * Save, version, and export trained PyTorch model artifacts to **Google Cloud Storage (GCS)** or **Hugging Face Hub** for deployment.

* **Automated Lineage & Artifact Tracking**:
  * Track datasets, hyperparameter configurations, and model versions together to guarantee end-to-end reproducibility.